In [2]:

# ================================================================
# JEEVAN LINK - ETA MODEL TRAINING
# Complete Dataset Loading, Training, Evaluation and Saving
# ================================================================

import pandas as pd
import numpy as np
import json
import joblib
import warnings

from pathlib import Path
from tkinter import Tk, filedialog

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer

from sklearn.ensemble import (
    RandomForestRegressor,
    ExtraTreesRegressor,
    HistGradientBoostingRegressor
)

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

warnings.filterwarnings("ignore")

# ================================================================
# 1. PROJECT PATHS
# ================================================================

PROJECT_DIR = Path(r"C:\Users\ishit\jeevan_link_ml")

MODEL_DIR = PROJECT_DIR / "Models" / "ETA_Model"

MODEL_DIR.mkdir(parents=True, exist_ok=True)

TARGET = "actual_travel_minutes"

RANDOM_STATE = 42

print("=" * 70)
print("JEEVAN LINK - ETA MODEL TRAINING")
print("=" * 70)


# ================================================================
# 2. SELECT AND LOAD DATASET
# ================================================================

print("\nPlease select your final ETA dataset.")

root = Tk()
root.withdraw()
root.attributes("-topmost", True)

selected_file = filedialog.askopenfilename(
    title="Select Final ETA Dataset",
    filetypes=[
        ("Dataset files", "*.csv *.xls *.xlsx"),
        ("All files", "*.*")
    ]
)

root.destroy()

if not selected_file:
    raise FileNotFoundError("No dataset was selected.")

DATA_PATH = Path(selected_file)

print("\nSelected file:", DATA_PATH)

# First try CSV, including CSV files saved with .xls extensions.
try:
    df = pd.read_csv(DATA_PATH)
    print("File loaded as CSV.")

except (UnicodeDecodeError, pd.errors.ParserError):
    df = pd.read_excel(DATA_PATH)
    print("File loaded as Excel.")

print("\nDATASET LOADED SUCCESSFULLY")
print("Shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())

if TARGET not in df.columns:
    raise ValueError(
        f"Target column '{TARGET}' not found.\n"
        f"Available columns: {df.columns.tolist()}"
    )


# ================================================================
# 3. DATASET QUALITY CHECK
# ================================================================

print("\n" + "=" * 70)
print("DATASET QUALITY CHECK")
print("=" * 70)

print("Missing values:", df.isnull().sum().sum())
print("Duplicate rows:", df.duplicated().sum())

if "request_id" in df.columns:
    print(
        "Duplicate request IDs:",
        df["request_id"].duplicated().sum()
    )

# Remove exact duplicate rows
df = df.drop_duplicates().copy()

# Convert target into numeric format
df[TARGET] = pd.to_numeric(
    df[TARGET],
    errors="coerce"
)

# Remove records without valid target
df = df.dropna(subset=[TARGET]).copy()

# Remove non-positive ETA values
df = df[df[TARGET] > 0].copy()

if len(df) < 30:
    raise ValueError("Not enough valid records to train the model.")

print("\nFinal dataset shape:", df.shape)

print("\nTARGET DISTRIBUTION")
print(df[TARGET].describe())


# ================================================================
# 4. FEATURE AND TARGET SEPARATION
# ================================================================

# Remove identifier from input features
X = df.drop(
    columns=[TARGET, "request_id"],
    errors="ignore"
).copy()

y = df[TARGET].copy()

# Remove columns that are entirely empty
empty_columns = X.columns[X.isnull().all()].tolist()

if empty_columns:
    print("\nRemoving completely empty columns:", empty_columns)
    X = X.drop(columns=empty_columns)

# Detect feature types
categorical_features = X.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

numerical_features = X.select_dtypes(
    include=["number"]
).columns.tolist()

unsupported_features = [
    col for col in X.columns
    if col not in categorical_features + numerical_features
]

if unsupported_features:
    raise ValueError(
        "Unsupported feature types found: "
        + str(unsupported_features)
    )

if X.shape[1] == 0:
    raise ValueError("No usable input features found.")

print("\nFEATURE INFORMATION")
print("Total features:", X.shape[1])
print("Numerical features:", len(numerical_features))
print("Categorical features:", len(categorical_features))

print("\nNumerical features:")
print(numerical_features)

print("\nCategorical features:")
print(categorical_features)


# ================================================================
# 5. TRAIN / VALIDATION / TEST SPLIT
# ================================================================

# 70% training, 15% validation, 15% testing

X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=RANDOM_STATE
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=RANDOM_STATE
)

print("\n" + "=" * 70)
print("DATA SPLIT")
print("=" * 70)

print("Training records:", len(X_train))
print("Validation records:", len(X_val))
print("Testing records:", len(X_test))


# ================================================================
# 6. PREPROCESSING PIPELINE
# ================================================================

numerical_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        )
    ]
)

categorical_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "encoder",
            OneHotEncoder(handle_unknown="ignore")
        )
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numerical_transformer,
            numerical_features
        ),
        (
            "categorical",
            categorical_transformer,
            categorical_features
        )
    ],
    remainder="drop"
)


# ================================================================
# 7. DEFINE REGRESSION MODELS
# ================================================================

models = {

    "Random Forest": RandomForestRegressor(
        n_estimators=300,
        max_depth=None,
        min_samples_split=2,
        min_samples_leaf=2,
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),

    "Extra Trees": ExtraTreesRegressor(
        n_estimators=300,
        max_depth=None,
        min_samples_split=2,
        min_samples_leaf=2,
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),

    "Hist Gradient Boosting": HistGradientBoostingRegressor(
        max_iter=200,
        learning_rate=0.08,
        max_leaf_nodes=20,
        l2_regularization=1.0,
        random_state=RANDOM_STATE
    )
}


# ================================================================
# 8. TRAIN AND VALIDATE MODELS
# ================================================================

validation_results = []
trained_models = {}

print("\n" + "=" * 70)
print("MODEL TRAINING STARTED")
print("=" * 70)

for model_name, model in models.items():

    print("\nTraining:", model_name)

    pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("regressor", model)
        ]
    )

    pipeline.fit(X_train, y_train)

    val_predictions = pipeline.predict(X_val)

    val_mae = mean_absolute_error(
        y_val,
        val_predictions
    )

    val_rmse = np.sqrt(
        mean_squared_error(y_val, val_predictions)
    )

    val_r2 = r2_score(
        y_val,
        val_predictions
    )

    validation_results.append({
        "model": model_name,
        "validation_MAE": val_mae,
        "validation_RMSE": val_rmse,
        "validation_R2": val_r2
    })

    trained_models[model_name] = pipeline

    print("Validation MAE:", round(val_mae, 4))
    print("Validation RMSE:", round(val_rmse, 4))
    print("Validation R2:", round(val_r2, 4))


# ================================================================
# 9. MODEL COMPARISON
# ================================================================

comparison_df = pd.DataFrame(validation_results)

comparison_df = comparison_df.sort_values(
    by="validation_MAE",
    ascending=True
).reset_index(drop=True)

comparison_df.to_csv(
    MODEL_DIR / "eta_model_comparison.csv",
    index=False
)

print("\n" + "=" * 70)
print("MODEL COMPARISON")
print("=" * 70)

print(comparison_df.to_string(index=False))

# Select best model using validation MAE
best_model_name = comparison_df.iloc[0]["model"]

print("\nBEST MODEL SELECTED:", best_model_name)


# ================================================================
# 10. RETRAIN BEST MODEL ON TRAINING + VALIDATION DATA
# ================================================================

X_train_full = pd.concat(
    [X_train, X_val],
    axis=0
)

y_train_full = pd.concat(
    [y_train, y_val],
    axis=0
)

best_model = models[best_model_name]

final_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("regressor", best_model)
    ]
)

print("\nRetraining selected model...")

final_pipeline.fit(
    X_train_full,
    y_train_full
)


# ================================================================
# 11. FINAL TEST EVALUATION
# ================================================================

test_predictions = final_pipeline.predict(X_test)

test_mae = mean_absolute_error(
    y_test,
    test_predictions
)

test_rmse = np.sqrt(
    mean_squared_error(y_test, test_predictions)
)

test_r2 = r2_score(
    y_test,
    test_predictions
)

print("\n" + "=" * 70)
print("FINAL TEST RESULTS")
print("=" * 70)

print("Selected model:", best_model_name)
print("Test MAE:", round(test_mae, 4), "minutes")
print("Test RMSE:", round(test_rmse, 4), "minutes")
print("Test R2:", round(test_r2, 4))


# ================================================================
# 12. SAVE TEST PREDICTIONS
# ================================================================

predictions_df = pd.DataFrame({
    "actual_travel_minutes": y_test.values,
    "predicted_travel_minutes": test_predictions,
    "absolute_error_minutes": np.abs(
        y_test.values - test_predictions
    )
})

predictions_df.to_csv(
    MODEL_DIR / "eta_test_predictions.csv",
    index=False
)


# ================================================================
# 13. SAVE MODEL METRICS
# ================================================================

metrics = {
    "model_name": best_model_name,
    "target": TARGET,
    "training_records": int(len(X_train_full)),
    "validation_records": int(len(X_val)),
    "test_records": int(len(X_test)),
    "total_features": int(X.shape[1]),
    "test_MAE_minutes": float(test_mae),
    "test_RMSE_minutes": float(test_rmse),
    "test_R2": float(test_r2),
    "random_state": RANDOM_STATE
}

with open(
    MODEL_DIR / "eta_model_metrics.json",
    "w"
) as f:
    json.dump(metrics, f, indent=4)


# ================================================================
# 14. SAVE TRAINED MODEL AND FEATURE INFORMATION
# ================================================================

joblib.dump(
    final_pipeline,
    MODEL_DIR / "eta_model.pkl"
)

joblib.dump(
    list(X.columns),
    MODEL_DIR / "eta_feature_names.pkl"
)

joblib.dump(
    TARGET,
    MODEL_DIR / "eta_target_name.pkl"
)

# Save the train/test split information for reproducibility
split_summary = {
    "training_records": int(len(X_train)),
    "validation_records": int(len(X_val)),
    "testing_records": int(len(X_test)),
    "random_state": RANDOM_STATE
}

with open(
    MODEL_DIR / "eta_data_split_summary.json",
    "w"
) as f:
    json.dump(split_summary, f, indent=4)


# ================================================================
# 15. SAMPLE PREDICTION
# ================================================================

print("\n" + "=" * 70)
print("SAMPLE ETA PREDICTION")
print("=" * 70)

sample_input = X_test.iloc[[0]]

sample_prediction = final_pipeline.predict(
    sample_input
)[0]

print(
    "Predicted ETA:",
    round(float(sample_prediction), 2),
    "minutes"
)

print(
    "Actual ETA:",
    round(float(y_test.iloc[0]), 2),
    "minutes"
)


# ================================================================
# 16. FINAL TRAINING SUMMARY
# ================================================================

print("\n" + "=" * 70)
print("ETA MODEL TRAINING COMPLETED SUCCESSFULLY")
print("=" * 70)

print("\nModel folder:")
print(MODEL_DIR)

print("\nSaved files:")

for file in sorted(MODEL_DIR.iterdir()):
    if file.is_file():
        print("-", file.name)

print("\nFINAL PERFORMANCE")
print("Best Model:", best_model_name)
print("MAE:", round(test_mae, 3), "minutes")
print("RMSE:", round(test_rmse, 3), "minutes")
print("R2:", round(test_r2, 4))

print("\nYour ETA model has been trained and saved!")

  


JEEVAN LINK - ETA MODEL TRAINING

Please select your final ETA dataset.

Selected file: C:\Users\ishit\jeevan_link_ml\Data\Processed_data\models\eta_model\eta_dataset_final (6).xls
File loaded as CSV.

DATASET LOADED SUCCESSFULLY
Shape: (5016, 29)

Columns:
['request_id', 'urgency_label', 'people_affected', 'people_trapped', 'injuries_reported', 'medical_emergency', 'water_level', 'building_condition', 'rescue_required', 'location_risk', 'context_peak_flood_level_m', 'context_event_duration_days', 'context_rainfall_t1d', 'context_percent_flooded_area', 'context_dfsi', 'flood_type', 'peak_discharge_q_cumec', 'flood_volume_cumec', 'time_to_peak_days', 'recession_time_day', 'drainage_area', 'catchment_relief', 'drainage_density', 'annual_precipitation', 'road_density', 'population_density', 'urban_percentage', 'synthetic_route_distance_km', 'actual_travel_minutes']

DATASET QUALITY CHECK
Missing values: 0
Duplicate rows: 0
Duplicate request IDs: 0

Final dataset shape: (5016, 29)

TARGET 